In [1]:
# This Python 3 environment comes with many helpful analytics libraries
###################
#This is section 1_ preprocessing of Assignment 2: Group Presentation 
#Step 1: Import libraries 
import numpy as np # linear algebra
import pandas as pd # data processing
import re #regular expressions 
from sklearn.compose import ColumnTransformer #for regular expressions to clean text data
from sklearn. preprocessing import OneHotEncoder, StandardScaler #for encoding categorical variables and scaling numerical features
from openpyxl import load_workbook #to load workbook

#Read the files' path 
PATH_2018 = r"C:\Users\ASUS\Downloads\Qualifications and work 2018-19 Data Tables (1).xlsx"
PATH_2022 = r"C:\Users\ASUS\Downloads\QAW2223DC (2).xlsx"

wb_2018 = load_workbook(filename=r"C:\Users\ASUS\Downloads\Qualifications and work 2018-19 Data Tables (1).xlsx", read_only=True)
wb_2022 = load_workbook(filename= r"C:\Users\ASUS\Downloads\QAW2223DC (2).xlsx", read_only=True)

#Count the sheet
print("Number of sheets:", len(wb_2018.sheetnames))
print("Sheet Names:", wb_2018.sheetnames)

print("Number of sheets:", len(wb_2022.sheetnames))
print("Sheet Names:", wb_2022.sheetnames)

Number of sheets: 27
Sheet Names: ['Contents', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5', 'Table 6', 'Table 7', 'Table 8', 'Table 9', 'Table 10', 'Table 11', 'Table 12', 'Table 13', 'Table 14', 'Table 15', 'Table 16', 'Table 17', 'Table 18', 'Table 19', 'Table 20', 'Table 21', 'Table 22', 'Table 23', 'Table 24', 'Table 25', 'Table 26']
Number of sheets: 18
Sheet Names: ['Contents', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5 ', 'Table 6', 'Table 7', 'Table 8 ', 'Table 9 ', 'Table 10', 'Table 11', 'Table 12 ', 'Table 13', 'Table 14', 'Table 15', 'Table 16', 'Table 17']


In [2]:
#Step 2: Create functions 
#Function 1: Use to clean extra space and blank cells 
def clean_cells(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip() #change the cell value into text and remove spaces
    text = re.sub(r"\s+", " ", text) #turn multiple spaces, or line breaks inside the text into normal space

    if text == "":
        return np.nan

    return text

#Function 2: Use to convert messy values into proper numbers and turn missing values into NaN
def to_number(value):
    if pd.isna(value):
        return np.nan

    return pd. to_numeric (
        str(value).strip().replace(",", "").lower(),
        errors="coerce"
    )

#Function 3: Use to check the rows instead of just the first cell
def rows_text(row, target_text):
    cleaned_value = [clean_cells(item) for item in row.tolist()]
    return target_text in cleaned_value

#Function 4,5,6: Use to normalise genders, job status and qualifications' names 
def cln_gender(label):
    label_map = {
        "Males": "Male",
        "Females": "Female",
        "Persons": "Persons"
    }
    return label_map.get(label, label)

def cln_jobstat(label):
    label_map = {
        "Employed full-time": "Full-time",
        "Employed part-time": "Part-time",
        "Total employed": "Total employed"
    }
    return label_map.get(label, label)

def cln_qualification_group(label):
    if label is None:
        return None
    
    label = clean_cells(label)
    
    aliases = {
        "Has one or more non-school qualifications": "Any_non_school_qualification",
        "One": "One",
        "One non-school qualification": "One",
        "Two": "Two",
        "Two non-school qualifications": "Two",
        "Three or more": "Three_or_more",
        "Three or more non-school qualifications": "Three_or_more",
        "No non-school qualification": "No_qualification",
    }
    return aliases.get(label, label)

print ("Cleaning and helper functions loaded successfully")

Cleaning and helper functions loaded successfully


In [3]:
#Function 7: Use to parse the income table
def parse_incometb(book_path, sheet_name, year_tag):
    raw_df = pd.read_excel(book_path, sheet_name=sheet_name, header=None)
    
    allowed_groups = {"One", "Two", "Three_or_more", "No_qualification"}
    job_labels = {"Employed full-time", "Employed part-time", "Total employed"}
    sex_labels = {"Males", "Females", "Persons"}
    
    parsed_rows = [] #store clean rows here 
    current_job = None
    current_sex = None

    qual_lookup = { #convert qualification groups into numeric code for modelling
        "No_qualification": 0,
        "One": 1,
        "Two": 2,
        "Three_or_more": 3
    }

    for _, row in raw_df.iterrows():
        if rows_text(row, "Proportion (%)"): #to stop when the table reaches the next section
            break 

        first_cell = clean_cells(row.iloc[0])

        if first_cell is None: 
            continue #to skip empty rows
            
        if first_cell in job_labels: 
            current_job = cln_jobstat(first_cell) #to update current employment stat
            current_sex = None #reset sex because a new job section has started 
            continue 

        if first_cell in sex_labels: 
            current_sex = cln_gender(first_cell) #to update current sex 
            continue 
    
        qual_group = cln_qualification_group(first_cell) #to convert first cell into qual groups 
    
        if qual_group not in allowed_groups:
            continue

        if current_job is None or current_sex is None:
            continue

        quintiles = [to_number(v) for v in row[1:7]]  #convert quintile and group size values to proper numbers
        avg_income = to_number(row[8] if len(row) > 8 else np.nan)  # convert average income to proper numbers
        median_income = to_number(row[9] if len(row) > 9 else np.nan)  #convert median income to proper numbers
    
        if pd.isna(avg_income):
            continue 
    
        parsed_rows.append({  # Add one cleaned row to the list
                "year": year_tag,
                "sex": current_sex,
                "employment_status": current_job,
                "qualification_group": qual_group,
                "qualification_count_code": qual_lookup[qual_group],
                "has_non_school_qualification": int(qual_group != "No_qualification"),
                "group_size_000": quintiles[5],
                "lowest_quintile_000": quintiles[0],
                "second_quintile_000": quintiles[1],
                "third_quintile_000": quintiles[2],
                "fourth_quintile_000": quintiles[3],
                "highest_quintile_000": quintiles[4],
                "avg_weekly_income": avg_income,
                "median_weekly_income": median_income
            }) 

    cleaned_df = pd.DataFrame(parsed_rows) #put clean rows into a dataframe

    if cleaned_df.empty: 
        raise ValueError("No rows were parsed")
    
    cleaned_df = cleaned_df[
        cleaned_df["sex"].isin(["Male", "Female"]) &
        cleaned_df["employment_status"].isin(["Full-time", "Part-time"])
    ].copy()

    cleaned_df["log_avg_weekly_income"] = np.log(cleaned_df["avg_weekly_income"])

    ordered_cols = [  #arranging variables into order
        "year", "sex", "employment_status", "qualification_group",
        "qualification_count_code", "has_non_school_qualification",
        "group_size_000", "lowest_quintile_000", "second_quintile_000",
        "third_quintile_000", "fourth_quintile_000", "highest_quintile_000",
        "avg_weekly_income", "median_weekly_income", "log_avg_weekly_income"
    ]

    return (
        cleaned_df[ordered_cols]  # keep only the ordered columns
        .sort_values(["employment_status", "sex", "qualification_count_code"])  # Sort rows clearly
        .reset_index(drop=True)  # reset row numbers
    )

print("Income parsing function loaded successfully")

Income parsing function loaded successfully


In [4]:
#Function 8: Use to build yearly datasets separately for training/ testing
def build_yearly_dataset():
    data_2018 = parse_incometb(PATH_2018, "Table 2", "2018")
    data_2022 = parse_incometb(PATH_2022, "Table 3", "2022")

    return data_2018, data_2022

#Function 9: Use to build training datasets by years 
def build_train_test_by_year(train_df, test_df):
    feature_cols = [
        "sex",
        "employment_status",
        "qualification_group",
        "qualification_count_code",
        "group_size_000"
    ]

    X_train_raw = train_df[feature_cols].copy() #to create training input data 
    y_train = train_df["log_avg_weekly_income"].copy() #to create the training target var
    
    X_test_raw = test_df[feature_cols].copy()
    y_test = test_df["log_avg_weekly_income"].copy()
    
    cat_cols = ["sex", "employment_status", "qualification_group"]
    num_cols = ["qualification_count_code", "group_size_000"]
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False) #to convert categoricals into 0/1 columns 

    #for categorical use OneHotEncoder, for num use StandardScaler 
    preprocessors = ColumnTransformer(
        transformers=[
            ("cat", encoder, cat_cols),
            ("num", StandardScaler(), num_cols)
        ]
    )
    X_train_ready = preprocessors.fit_transform(X_train_raw)
    X_test_ready = preprocessors.transform(X_test_raw)

    feature_names = preprocessors.get_feature_names_out()


    X_train_ready_df = pd.DataFrame(X_train_ready, columns=feature_names)
    X_test_ready_df = pd.DataFrame(X_test_ready, columns=feature_names)

    return (
        X_train_raw,
        y_train,
        X_test_raw,
        y_test,
        X_train_ready_df,
        X_test_ready_df,
        preprocessors
    )

#Function 10: Run preprocessing and display results 
def run_preprocessing():
    data_2018, data_2022 = build_yearly_dataset()
    (
        X_train_raw,
        y_train,
        X_test_raw,
        y_test,
        X_train_ready_df,
        X_test_ready_df,
        preprocessor
    ) = build_train_test_by_year(train_df= data_2022, test_df= data_2018)

    
    out_2018 = r"C:\Users\ASUS\Downloads\regression_dataset_2018.csv"
    out_2022 = r"C:\Users\ASUS\Downloads\regression_dataset_2022.csv"
    x_train_out = r"C:\Users\ASUS\Downloads\X_train_raw_2022.csv"
    y_train_out = r"C:\Users\ASUS\Downloads\y_train_2022.csv"
    x_test_out = r"C:\Users\ASUS\Downloads\X_test_raw_2018.csv"
    y_test_out = r"C:\Users\ASUS\Downloads\y_test_2018.csv"
    x_train_ready_out = r"C:\Users\ASUS\Downloads\X_train_processed_2022.csv"
    x_test_ready_out = r"C:\Users\ASUS\Downloads\X_test_processed_2018.csv"
    
    data_2018.to_csv(out_2018, index=False)
    data_2022.to_csv(out_2022, index=False)
    
    X_train_raw.to_csv(x_train_out, index=False)
    y_train.to_csv(y_train_out, index=False)
    X_test_raw.to_csv(x_test_out, index=False)
    y_test.to_csv(y_test_out, index=False)
    
    X_train_ready_df.to_csv(x_train_ready_out, index=False)
    X_test_ready_df.to_csv(x_test_ready_out, index=False)

    print("2018 dataset shape:", data_2018.shape)
    print("2022 dataset shape:", data_2022.shape)
    print("2018 preview:")
    print(data_2018.head(8))
    print(data_2022.head(8))
    print("Training year: 2022")
    print("Testing year: 2018")
    print("X_train_raw shape:", X_train_raw.shape)
    print("X_test_raw shape:", X_test_raw.shape)
    print("X_train_processed shape:", X_train_ready_df.shape)
    print("X_test_processed shape:", X_test_ready_df.shape)
    print("y_train shape:", y_train.shape)
    print("y_test shape:", y_test.shape)
    
    return (
        data_2018,
        data_2022,
        X_train_raw,
        y_train,
        X_test_raw,
        y_test,
        X_train_ready_df,
        X_test_ready_df,
        preprocessor
    )
    

In [5]:
results = run_preprocessing()

(
    data_2018,
    data_2022,
    X_train_raw,
    y_train,
    X_test_raw,
    y_test,
    X_train_ready_df,
    X_test_ready_df,
    preprocessor
) = results

display(data_2018)
display(data_2022)


2018 dataset shape: (16, 15)
2022 dataset shape: (16, 15)
2018 preview:
   year     sex employment_status qualification_group  \
0  2018  Female         Full-time    No_qualification   
1  2018  Female         Full-time                 One   
2  2018  Female         Full-time                 Two   
3  2018  Female         Full-time       Three_or_more   
4  2018    Male         Full-time    No_qualification   
5  2018    Male         Full-time                 One   
6  2018    Male         Full-time                 Two   
7  2018    Male         Full-time       Three_or_more   

   qualification_count_code  has_non_school_qualification  group_size_000  \
0                         0                             0           741.2   
1                         1                             1          1333.0   
2                         2                             1           774.7   
3                         3                             1           320.5   
4                         0  

,year,sex,employment_status,qualification_group,qualification_count_code,has_non_school_qualification,group_size_000,lowest_quintile_000,second_quintile_000,third_quintile_000,fourth_quintile_000,highest_quintile_000,avg_weekly_income,median_weekly_income,log_avg_weekly_income
0,2018,Female,Full-time,No_qualification,0,0,741.2,17.0,16.0,215.0,288.1,123.6,1142.1,1036.0,7.040624
1,2018,Female,Full-time,One,1,1,1333.0,19.6,27.5,254.6,463.4,403.4,1388.5,1247.0,7.235979
2,2018,Female,Full-time,Two,2,1,774.7,5.0,7.6,93.6,292.5,304.2,1584.0,1438.0,7.367709
3,2018,Female,Full-time,Three_or_more,3,1,320.5,4.1,5.8,29.7,92.9,164.8,1750.9,1592.0,7.467885
4,2018,Male,Full-time,No_qualification,0,0,1423.5,29.9,44.1,318.0,492.3,391.0,1304.1,1151.0,7.173268
5,2018,Male,Full-time,One,1,1,2443.0,34.1,35.5,261.9,763.5,1063.7,1711.4,1500.0,7.445067
6,2018,Male,Full-time,Two,2,1,997.5,17.3,8.5,72.1,256.1,559.2,1993.8,1726.0,7.597798
7,2018,Male,Full-time,Three_or_more,3,1,390.2,8.2,6.6,36.1,69.2,236.7,2352.2,1918.0,7.763106
8,2018,Female,Part-time,No_qualification,0,0,931.7,252.8,216.1,257.6,69.9,10.8,491.7,426.9,6.197869
9,2018,Female,Part-time,One,1,1,1014.2,103.7,163.3,383.3,152.0,66.1,757.4,671.0,6.629892


,year,sex,employment_status,qualification_group,qualification_count_code,has_non_school_qualification,group_size_000,lowest_quintile_000,second_quintile_000,third_quintile_000,fourth_quintile_000,highest_quintile_000,avg_weekly_income,median_weekly_income,log_avg_weekly_income
0,2022,Female,Full-time,No_qualification,0,0,706.7,24.4,38.6,223.6,237.8,72.8,1274.1,1151.0,7.149995
1,2022,Female,Full-time,One,1,1,1747.7,35.7,38.0,332.3,639.0,487.6,1691.7,1496.0,7.433489
2,2022,Female,Full-time,Two,2,1,845.8,11.1,11.6,159.5,270.4,299.3,1836.4,1668.0,7.515562
3,2022,Female,Full-time,Three_or_more,3,1,357.6,5.1,0.0,51.9,106.2,160.7,2130.9,1822.0,7.664300
4,2022,Male,Full-time,No_qualification,0,0,1351.8,26.2,58.3,314.2,439.4,314.8,1574.3,1342.0,7.361566
5,2022,Male,Full-time,One,1,1,2885.2,31.2,55.8,366.8,855.6,1119.3,2029.9,1726.0,7.615742
6,2022,Male,Full-time,Two,2,1,1085.9,23.4,21.5,90.0,287.2,542.1,2327.6,1918.0,7.752593
7,2022,Male,Full-time,Three_or_more,3,1,471.4,5.5,6.3,41.5,107.3,268.5,2557.7,2110.0,7.846864
8,2022,Female,Part-time,No_qualification,0,0,937.0,290.6,208.8,238.0,58.9,10.1,569.4,500.0,6.344583
9,2022,Female,Part-time,One,1,1,1169.0,111.0,207.8,406.6,192.6,59.0,914.5,800.0,6.818377
